# Analysis Overview

Central entry point for all analysis notebooks.
Core questions, notebook index and key findings summary.

## Core Questions

1. Where do delays occur? — stops, districts, lines
2. When do delays occur? — time of day, weekday, season
3. What amplifies delays? — weather, events
4. What does the target itself look like? — distribution, OTP, arr vs dep
5. Which features correlate most with delays?
6. Can delays be predicted? → modeling

## Notebooks

| Notebook | Focus |
|:---|:---|
| `03_analysis_1-target.ipynb` | Delay distribution, OTP, arr vs dep, cancellations |
| `03_analysis_2-network.ipynb` | Netzveränderungen 2023–2025 · Vor/Nachher · Einlaufzeit · Hotspots · Versorgungsqualität |
| `03_analysis_3-temporal.ipynb` | Hour · weekday · month · season · full year |
| `03_analysis_4-spatial.ipynb` | Stops · districts · lines |
| `03_analysis_5-meteo.ipynb` | Rain · wind · snow · temperature |
| `03_analysis_6-events.ipynb` | Holidays · events · event size |

> **Hinweis für alle Notebooks:** Das Tramnetz hat sich im Analysezeitraum 2023–2025 verändert.
> Fahrplanwechsel Dezember 2023 (j23 → j24): Linien 9, 11 und 13 wurden fundamental umgebaut.
> Bei linienbezogenen Befunden immer `03_analysis_2-network.ipynb` als Kontext heranziehen.

## Line Colors

Offizielle VBZ-Linienfarben aus GTFS `routes.txt` — verfügbar via `line_color("12")` aus `zh_tram_flow.config`.

| Linie | Farbe | | Linie | Farbe | | Linie | Farbe |
|:---:|:---|:---|:---:|:---|:---|:---:|:---|
| **2** | 🟥 `#E20A16` | | **8** | 🟩 `#8AB51F` | | **14** | 🟦 `#008DC5` |
| **3** | 🟩 `#00892F` | | **9** | 🟦 `#11296F` | | **15** | 🟥 `#E20A16` |
| **4** | 🟦 `#11296F` | | **10** | 🟪 `#E12472` | | **17** | 🟥 `#8E224D` |
| **5** | 🟫 `#734522` | | **11** | 🟩 `#00892F` | | **19** | 🟥 `#E20A16` |
| **6** | 🟧 `#CA7D3C` | | **12** | 🩵 `#92D6E3` | | **E** | 🟥 `#E20A16` |
| **7** | ⬛ `#000000` | | **13** | 🟨 `#FFCC00` | | | |

## Key Findings

> Alle Findings werden in den jeweiligen Analysis-Notebooks erarbeitet und hier zentral gelistet.
> ID-Schema: `F-{NOTEBOOK}-{NR}` — Status: `open` · `in-progress` · `done` · `⚠️ aktiv`

| ID | Notebook | Section | Finding | Impact | Action | Action Location | Status |
|:---|:---|:---|:---|:---|:---|:---|:---|
| F-TARGET-01 | target | Delay Distribution | `arrival_delay` rechtsschiefe Verteilung — Median ≠ Mean | Lineare Modelle unterschätzen Extremwerte | Log-Transform + MdAE-Robustness-Check | `03_analysis_1-target` | done |
| F-TARGET-02 | target | Delay Delta | `delay_delta` bimodal — Terminus-Cluster bei −50s, kein Datenfehler | Systematisches Signal für Starthaltestellen | Terminus-Flag als Feature ableiten | `03_analysis_4-spatial` | done |
| F-TARGET-03 | target | OTP | 71.2% der Trips pünktlich (|arr_delay| ≤ 120s); Median Delay positiv → systemischer Puffermangel | Fahrplan strukturell zu knapp | Dwell-Time-Feature aus Preparation | `02_preparation` | done |
| F-TARGET-04 | target | Target Definition | `dep_schedule − arr_schedule` = dwell_time verfügbar; 71.3% = 0s (Durchfahrten ohne Haltezeit) | Feature weniger aussagekräftig als erwartet — binäres `has_dwell` prüfen | `dwell_time` + `has_dwell` in Preparation | `02_preparation` | done |
| F-TARGET-05 | target | Cancellations | `canceled`-Artefakt vor Jul 2024 (Datendefinitionsänderung) — kein reales Betriebsproblem. Cancellation Rate effektiv: 6.2% | Verfälscht Cancellation-Baseline | `canceled=True` aus Delay-Modell; `is_pre_july_2024` Feature | `02_preparation` | done |
| F-TARGET-06 | target | Monthly Trend | Nov–Dez 2025 GTFS-Artefakt: −0.8s Verzerrung (−1.5%) — marginal | Minimaler Einfluss; Bereinigung trotzdem sauber | Nov–Dez 2025 aus Train+Test entfernen | `02_preparation` | ⚠️ aktiv |
| F-TARGET-07 | target | Extreme Values | Extremwerte bis +5000s vorhanden — potenzielle Messfehler oder Störungsereignisse | Robust-Modell bevorzugen | Abgleich mit Wetter- und Event-Daten | `03_analysis_3-temporal` | open |
| F-TARGET-08 | target | OTP | `trip_id` + `stop_sequence` im Master-Datensatz — Kaskadenanalyse möglich | Prediction-Signal verfügbar | `trip_id` in Feature Engineering | `02_preparation` | done |
| F-TARGET-09 | target | Trend | Bereinigter Delay-Delta Trend: j23=+4.3s → j24=+4.9s → j25=+5.0s (organisch, nicht dramatisch) | Struktureller Aufwärtstrend real aber moderat | `year` + `month` als Features | Modell-Phase | ⚠️ aktiv |
| F-TARGET-10 | target | Trend | `arrival_delay` 2025 (Jan–Okt): 55.1s — leicht unter 2024 (58.5s) → Stabilisierung im Ankunfts-Delay | Netz wird nicht in allen Metriken schlechter | In Trend-Analyse hervorheben | `03_analysis_3-temporal` | done |
| F-TARGET-11 | target | Cancellations | Synchrone Cancellation-Erhöhung aller Linien vor Jul 2024 ohne gleichzeitigen Delay-Anstieg — beweist Datendefinitions-Änderung | Stärkstes Qualitäts-Argument | `is_pre_july_2024` als Pflicht-Feature | `02_preparation` | done |
| F-TARGET-12 | target | Line Outlier | **Linie E**: OTP 56.2%, Ø Delay 128–130s — Sonderlinie/Entlastungslinie, kein Datenfehler; im Modell als eigene Klasse behandeln | Starker Outlier verzerrt Modell-Baseline | Als Sonderlinie annotieren; ggf. separates Modell | Modell-Phase | done |
| F-NET-01 | network | Netzveränderungen | Im GTFS zeigen L9/L11/L13 Dez 2023 markante Haltestellen-Zunahmen (+8/+13/+19). Neue Halte konzentrieren sich auf **Innenstadt** (Kreis 1: 12 neue Halte) — nicht auf Peripherie. Wahrscheinlich Flexity-Rollout/GTFS-Artefakt. | Jahresvergleiche nur mit Kontext möglich | `gtfs_year` Feature kodiert Zeitschnitt | `02_preparation` | done |
| F-NET-02 | network | Netzveränderungen | Stabile Referenzlinien: L10, L12, L14, L17 identisch über alle Jahre. Anomalie L2: 31→21→31 Halte (GTFS-Routing-Variante). | Kontrollgruppe für Zeitreihenvergleiche | Als Referenzlinien verwenden | `03_analysis_3-temporal` | done |
| F-NET-03 | network | Feature | `gtfs_year` erklärt netzweit nur +0.5s (j23=55.9s vs j24_j25=56.4s) — schwaches Feature | Kein klarer Netzwechsel-Effekt im Delay | `n_stops_line` als Alternative testen | `02_preparation` | done |
| F-NET-04 | network | Einlaufzeit | **Kein Einlaufzeit-Effekt**: L11 neue Halte −7.3s besser als bestehende, L9 −6.6s besser. Kein einheitlicher Trend über alle Linien. | Neubaustrecken benötigen keine Einlaufzeit-Toleranz | Feature `is_new_stop` wenig aussagekräftig | `03_analysis_2-network` | done |
| F-NET-05 | network | Hotspots | **Keine Korrelation** Linienanzahl × Delay: 0 Overlap. Central (15 Linien, 48.3s), Paradeplatz (14 Linien, 48.2s) — beide unter Netzschnitt (~56s). Kaskadenrisiko-Hypothese widerlegt. | `n_lines_at_stop` schwaches Feature | Aussenkorridore statt Knotenpunkte als Hotspots | `03_analysis_4-spatial` | done |
| F-NET-06 | network | Versorgungsqualität | Kreis 12 (+2) und Kreis 4 (+2) gewinnen neue Linienanbindungen. Kreis 7 verliert 2 Linien. Kreis 1 erhielt meisten neue Halte (12) aber keine neuen Linien. | Räumliche Netzstruktur-Änderung dokumentiert | Kreise 12/4 als strukturell verbessert; Kreis 7 verschlechtert | `03_analysis_2-network` | done |
| F-NET-07 | network | Kaskaden | `trip_id` ermöglicht Kaskadenanalyse — `prev_trip_delay` als Feature | Prediction-Signal | `trip_id` in Feature Engineering | `02_preparation` | open |
| F-NET-08 | network | Linie E | Linie E: 128–130s Ø Delay, OTP 56% — Entlastungslinie im GTFS, kein Datenfehler | Extremer Outlier — separat behandeln | Als Sonderlinie annotieren (vgl. F-TARGET-12) | Modell-Phase | done |
| F-TEMP-01 | temporal | Stunden-Profil | Kein klassischer Morgenrush: 7h=48.9s liegt **unter** Ø. Dominantes Muster: Nachmittag/Abend steigt ab 14h, Peak bei **21h=67.9s** (Events-Abreisewelle), starker Abend-Peak 17h=65.2s | `hour` ist stärkstes temporales Feature | `hour` + `hour × has_event` als Feature | `02_preparation` | done |
| F-TEMP-02 | temporal | Wochentag | **Donnerstag** kritischster Wochentag: Ø 60.4s, P95=194s. Montag (52.3s) und Sonntag (48.4s) beste Tage. | `weekday` als Feature | `day_of_week` ordinalkodiert | `02_preparation` | done |
| F-TEMP-03 | temporal | Wochentag | Donnerstag-Peak vereinbar mit Events-Häufung (Do-Abend) und HO-Hypothese — nicht direkt belegt | Interaktion Do × Abend × Events prüfen | `is_school_week` als Modifier | `02_preparation` | done |
| F-TEMP-04 | temporal | Wochentag | Samstag (57.0s) kaum besser als Werktag; Sonntag (48.4s) deutlich besser | `weekday` ordinalkodiert statt binär `is_weekend` | `day_of_week` 7-Kategorien | `02_preparation` | done |
| F-TEMP-05 | temporal | Monat | **November-Peak**: Nov 2023=67.9s, Nov 2024=72.9s — jeweils Jahreshöchstwert, ca. 10–12s über restlichem Jahr | `is_november` als Feature-Flag | Ursache: Laub + Baustellensaison + MIV | `02_preparation` | done |
| F-TEMP-06 | temporal | Saison | Herbst=61.2s (schlechteste Jahreszeit, OTP 85.2%), **Winter=51.7s (BESTE Jahreszeit, OTP 88.9%)** — überraschend | `season` als Feature | Winter-Vorteil: MIV-Reduktion überkompensiert Witterung | `02_preparation` | done |
| F-TEMP-07 | temporal | Zeitreihe | Struktureller Aufwärtstrend: 2024 Frühling/Sommer +4–7s über 2023; 2025 Jan–Okt leicht moderater (Stabilisierung) | `year` + `month` als Features | Rolling-Baseline als Feature-Idee | `02_preparation` | done |
| F-TEMP-08 | temporal | Zeitreihe | Schulferien-Täler erkennbar im Rolling-Average — Schulferien-Flag als Feature-Kandidat | Ferieneffekt additiv zu Wochentag | `is_school_holiday` aus ZH-Schulkalender | `02_preparation` | done |
| F-TEMP-09 | temporal | Feature | `gtfs_year` erklärt netzweit nur +0.5s — schwaches Feature. Linienebene: L11 +5.5s, L9 −4.3s — keine einheitliche Richtung. | Zeitvariable, nicht Netzstruktur-Feature | Bestätigt F-NET-03 | `02_preparation` | done |
| F-SPAT-01 | spatial | Hotspots | Delay-Hotspots sind NICHT zentrale Knotenpunkte, sondern **periphere Aussenkorridore**: Friedhof Enzenbühl 93.8s, Balgrist 85.2s, Leutschenbach 82.7s. Top-2 (Bertastrasse 181.6s, n=1'307) statistisch instabil. | `stop_name` stärkster räumlicher Prädiktor | Target-Encoding + n-Threshold-Filter | `02_preparation` | done |
| F-SPAT-02 | spatial | Terminus | Terminus-Haltestellen zeigen negative Delay-Werte (Frühankünfte); Start-Stop-Proxy findet 0 Kandidaten mit gewählten Schwellwerten | Kein messbarer Verzerrungseffekt (0.0s) | n-Threshold-Filter statt is_start_stop | `02_preparation` | done |
| F-SPAT-03 | spatial | Stadtkreise | **Kreis 11 schlechtester** (68.3s, OTP 83%), Kreis 12 (66.3s), Kreis 8 (63.7s). Innenstadt Kreis 1=51.3s (gut). **Kreis 5 bester** (49.9s, OTP 89%). | `district_nr` additiv nützlich | Kreise 11/12 als High-Risk-Marker | `02_preparation` | done |
| F-SPAT-04 | spatial | Linien-Ranking | **Alle Linien** haben positives Delay-Delta (akkumulieren, keine baut ab). Stärkste Akkumulatoren L10 (+6.5s), L11 (+6.2s), L4 (+8.1s). L51 (+20.2s) trotz niedrigstem Delay. | `line_name` stärkster räumlicher Prädiktor | Linien-Encoding als Feature | `02_preparation` | done |
| F-SPAT-05 | spatial | Features | `line_name` stärkster räumlicher Prädiktor. **L11** (68.7s, OTP 82%) kritischste Hauptlinie. `district_nr` additiv. | Beide Features ins Modell | Target-Encoding für `stop_name` | `02_preparation` | done |
| F-SPAT-06 | spatial | Starthaltestellen | Starthaltestellen-Proxy findet 0 Kandidaten — Verzerrung 0.0s. Low-Volume-Stops (n=1'307 für Top-Delay-Ausreisser) problematischer. | n-Threshold-Filter empfohlen | `n_threshold` in Modell-Preprocessing | `02_preparation` | done |
| F-SPAT-07 | spatial | Linien-Dichte | **0 Overlap** Top-20-Linienanzahl × Top-20-Delay: Haldenegg (15 Linien, 44.5s), Paradeplatz (14 Linien, 48.2s) — alle unter Netzschnitt. Kaskadenrisiko-Hypothese widerlegt. | `n_lines_at_stop` schwaches Feature | Aussenkorridore statt Knotenpunkte | `02_preparation` | done |
| F-SPAT-08 | spatial | dwell_time | `dwell_time` = 0s für **71.3%** aller Halte (Median=0 für alle Linien) — kein klarer Zusammenhang mit Delay messbar. Binäres `has_dwell` als Alternative. | Feature schwächer als erwartet | `has_dwell` testen | `02_preparation` | done |
| F-WEAT-01 | weather | Schnee | **Schnee stärkster Wettereffekt**: +54.0s, OTP 87.1%→76.1% (−10.9pp) | `has_snow` wichtigstes Wetter-Feature | `has_snow` priorisieren | `02_preparation` | done |
| F-WEAT-02 | weather | Regen | Starkregen: +23.3s, OTP −7.6pp. Regen: +8.9s, −3.1pp. Dosis-Wirkungs: <2mm=62.6s → >10mm=89.5s | `precipitation` kontinuierlich als Feature | Beide Repräsentationen ins Modell | `02_preparation` | done |
| F-WEAT-03 | weather | Wind | `is_windy` = NaN überall — Feature nie korrekt befüllt; Zürich windgeschützt, Trams schwer → kaum Effekt erwartet. **`is_windy` aus Feature-Set entfernt.** | Feature-Set bereinigt | `is_windy` entfernt | `02_preparation` | done |
| F-WEAT-04 | weather | Temperatur | **Kälte BESSER**: 0–5°C=53.8s (niedrigster Delay). Wärme schlechter: 25–30°C=59.7s. `is_hot` (>20°C) = +2.0s Delta — schwaches aber reales Signal. | Kleiner Effekt; Frost-Hypothese falsch | `temperature` kontinuierlich + `is_hot` Flag | `02_preparation` | done |
| F-WEAT-05 | weather | Multikollinearität | Alle Wetter-Features haben sehr niedrige Korrelation mit `arrival_delay` (max 0.042). Wetter × Saison nahezu unkorreliert (<0.05) — kein Multikollinearitätsproblem. | Wetter sind schwache aber eigenständige Signale | Alle Wetter-Features behalten | `02_preparation` | done |
| F-WEAT-06 | weather | Features | `precipitation` (r=0.036) und `has_snow` (r=0.038) nützlichste Wetter-Features; `temperature` (r=0.018) schwächer | Priorität: Schnee + Niederschlag | Feature-Selection nach Modell-Training | `02_preparation` | done |
| F-EVNT-01 | events | Feiertage | Feiertage **46.3s vs. Normal 56.2s (−9.9s, OTP +3.6pp)** — `is_holiday` als stärkstes einzelnes Event-Feature mit negativem Vorzeichen | `is_holiday` wichtigstes Event-Feature | `is_holiday` in `02_preparation` (bereits vorhanden) | `02_preparation` | done |
| F-EVNT-02 | events | Event-Grösse | Event-Skalierung: Gross=66.7s (+10.5s), Mittel=58.9s (+2.7s), **Klein=56.2s (+0.05s ≈ Normal)**. Event-Gewicht 1 hat keine Vorhersagekraft. | `event_weight` ordinales Feature; Klasse 1 binarisieren | `event_weight ≥ 2` als Schwelle prüfen | `02_preparation` | done |
| F-EVNT-03 | events | Stunden-Profil | Event-Effekt primär **Abend-Phänomen (18–22h)** — tagsüber kein Unterschied. Erklärt 21h-Spike aus F-TEMP-01. | `has_event × hour` Interaktion wichtiger als Haupteffekt | `event_weight × hour` Interaction als Feature | `02_preparation` | done |
| F-EVNT-04 | events | Event-Typ | **Fachmessen schlechteste Kategorie** (66.0s, OTP 84%) — nicht Konzerte. Konzerte 61.4s. Super League 53.8s nahe Normal. | `event_type` encoding; Fachmessen-Effekt für L11 relevant | `event_type` kategorisch | `02_preparation` | done |
| F-EVNT-05 | events | Klassenungleichgewicht | Gross-Events n=724k vs. Normal 70.5M — stark unbalanced. `is_holiday` stärkstes negatives Signal. | Oversampling oder gewichtetes Training | Event-Strategie in Modell-Phase | Modell-Phase | done |
| F-EVNT-06 | events | Stadtkreise | Stadtkreis-Δ auf Event-Tagen minimal (max +3.0s Kreis 2). Kreis 11 (Hallenstadion) überraschend −0.8s — räumliche Aggregation verbirgt Abend-Effekt. | `has_event × hour` Interaktion aussagekräftiger als Kreis × Event | Abend-Fokus in Feature Engineering | `02_preparation` | done |

## Kernfragen & KPIs — Beantwortbarkeit

> Stand nach Analyse-Phase (vor Modellierung). ✅ = bereits beantwortbar · 🔜 = benötigt Modell · ⚠️ = teilweise / Hypothese

---

### Baseline KPIs (aus Analyse)

| KPI | Wert | Quelle |
|:---|:---|:---|
| OTP (arrival_delay ≤ 120s) | 87.0% | F-TARGET — Schwellwert ±120s = VBZ-Standard / VDPW |
| Ø arrival_delay (2025 Jan–Okt, bereinigt) | 55.1s | F-TARGET-10 |
| Ø delay_delta (2025 Jan–Okt, bereinigt) | ~+5.0s | F-TARGET-09 |
| Cancellation Rate (effektiv, ab Jul 2024) | 6.2% | F-TARGET-05 |
| November-Anomalie (Artefakt ohne Bereinigung) | +17s / +26s delta | F-TARGET-06 |
| Aufwärtstrend delay_delta 2023→2025 | +4.3s → +4.9s → +5.0s | F-TARGET-09 |

---

### Kernfrage 1 — Wo entstehen Verspätungen?

**✅ Qualitativ beantwortbar.** Räumliche Hotspots und Linien-Ranking aus Analyse-Phase bekannt.

| Teilfrage | Status | Findings |
|:---|:---|:---|
| Welche Haltestellen sind die grössten Hotspots? | ✅ Periphere Aussenkorridore: Friedhof Enzenbühl 93.8s, Balgrist 85.2s, Leutschenbach 82.7s — NICHT zentrale Knotenpunkte | F-SPAT-01, F-SPAT-07 |
| Welche Linien akkumulieren Verspätung? | ✅ L11 (68.7s, OTP 82%) kritischste Hauptlinie; alle Linien akkumulieren positiv | F-SPAT-04, F-SPAT-05 |
| Welche Stadtkreise haben die höchsten Delays? | ✅ Kreis 11 (68.3s, OTP 83%) schlechtester; Kreis 12 (66.3s); Kreis 5 (49.9s, OTP 89%) bester | F-SPAT-03 |
| Wie gross ist der Starthaltestellen-Verzerrungseffekt? | ✅ 0.0s — kein messbarer Effekt; 0 Starthaltestellen-Kandidaten gefunden | F-SPAT-06, F-SPAT-02 |
| Sind Liniendichte und Verspätung korreliert? | ✅ Keine Korrelation — 0 Overlap; Central (15 Linien, 48.3s) und Paradeplatz (14 Linien, 48.2s) unter Netzschnitt | F-SPAT-07, F-NET-05 |

---

### Kernfrage 2 — Wann entstehen Verspätungen?

**✅ Klar beantwortbar.** Zeitliche Muster vollständig analysiert.

| Teilfrage | Status | Findings |
|:---|:---|:---|
| Welche Tagesstunden sind kritisch? | ✅ Kein klassischer Morgenrush (7h=48.9s unter Ø); Peak 21h=67.9s (Events-Abreisewelle); Abend-Peak 17h=65.2s | F-TEMP-01 |
| Welcher Wochentag ist am schlechtesten? | ✅ Donnerstag (60.4s, P95=194s); Montag (52.3s) und Sonntag (48.4s) beste Tage | F-TEMP-02 |
| Welcher Monat ist am schlechtesten? | ✅ November (Nov 2023=67.9s, Nov 2024=72.9s — jeweils Jahreshöchstwert) | F-TEMP-05 |
| Welche Jahreszeit ist am schlechtesten? | ✅ Herbst (61.2s, OTP 85.2%); Winter überraschend beste Jahreszeit (51.7s, OTP 88.9%) | F-TEMP-06 |
| Gibt es einen Aufwärtstrend? | ✅ Ja, moderat und strukturell; 2024 Frühling/Sommer +4–7s über 2023 | F-TARGET-09, F-TEMP-07 |
| Ist der Schulferien-Effekt messbar? | ⚠️ sichtbar im Rolling-Average, noch nicht quantifiziert | F-TEMP-08 |

---

### Kernfrage 3 — Was verstärkt Verspätungen?

**✅ Qualitativ beantwortbar.** Effektrichtung und relative Grösse bekannt.

| Einflussfaktor | Effekt | Status | Findings |
|:---|:---|:---|:---|
| Schnee | stark positiv (+54.0s, OTP −10.9pp) | ✅ stärkster Wettereffekt | F-WEAT-01 |
| Starkregen | moderat positiv (+23.3s); skaliert mit Intensität | ✅ Dosis-Wirkungs-Effekt | F-WEAT-02 |
| Wind | minimal / unklar | ⚠️ `is_windy` = NaN — Feature fehlt oder ohne Varianz | F-WEAT-03 |
| Hohe Temperatur (>20°C) | schwach positiv (+2.0s); Kälte (0–5°C) ist BESTE Bedingung (53.8s) | ✅ Frost-Hypothese falsch | F-WEAT-04 |
| Feiertag | stark negativ (−9.9s, OTP +3.6pp) — bester Tag-Typ | ✅ Berufsverkehr-Reduktion überwiegt | F-EVNT-01 |
| Grosse Events | positiv (+10.5s), primär Abend-Phänomen (18–22h) | ✅ Fachmessen schlechteste Kategorie (66.0s); Klein-Events ≈ Normal | F-EVNT-03, F-EVNT-04 |
| Fahrplanwechsel j23→j24 | strukturell; netzweit nur +0.5s | ✅ 3 Linien fundamental umgebaut | F-NET-01, F-NET-03 |

---

### Kernfrage 4 — Wie sieht das Ziel selbst aus?

**✅ Vollständig beantwortbar.**

| Aspekt | Antwort | Findings |
|:---|:---|:---|
| Verteilungsform | Rechtsschiefe (Long Tail) — Log-Transform prüfen | F-TARGET-01 |
| Bimodalität | delay_delta bimodal (Terminus-Cluster) — kein Fehler | F-TARGET-02 |
| Datenqualität Cancellations | Artefakt vor Jul 2024 — Datendefinitions-Änderung; effektive Rate 6.2% | F-TARGET-05, F-TARGET-11 |
| Datenqualität Nov/Dez 2025 | GTFS-Artefakt (−0.8s, −1.5%) — marginal aber bereinigen | F-TARGET-06 |
| Arrival vs. Departure | `delay_delta` = netto akkumuliert (+4.3s→+5.0s Trend); `arrival_delay` = Gesamtpuffer | F-TARGET-03, F-TARGET-04 |
| Linie E Outlier | OTP 56.2%, Ø 128–130s — Entlastungslinie, separat behandeln | F-TARGET-12, F-NET-08 |

---

### Kernfrage 5 — Welche Features korrelieren am stärksten?

**⚠️ Teilweise beantwortbar.** Qualitatives Ranking aus Analyse, quantitativ erst nach Modell-Training.

| Feature-Gruppe | Erwartete Stärke | Basis |
|:---|:---|:---|
| `hour` | ⭐⭐⭐ hoch | F-TEMP-01 — konsistentester Effekt (21h=+11.7s über Ø) |
| `stop_name` / `line_name` | ⭐⭐⭐ hoch | F-SPAT-01, F-SPAT-05 — L11 68.7s vs. L-Mittel |
| `day_of_week` | ⭐⭐ mittel | F-TEMP-02 — Do 60.4s vs. So 48.4s |
| `has_snow` | ⭐⭐ mittel (saisonal) | F-WEAT-01 — r=0.038, +54s absolut |
| `event_weight × hour` | ⭐⭐ mittel (Abend) | F-EVNT-03 — Interaktion wichtiger als Haupteffekt |
| `month` / `season` | ⭐⭐ mittel | F-TEMP-05/06 — Nov peak, Winter best |
| `is_holiday` | ⭐⭐ mittel (negativ) | F-EVNT-01 — stärkstes negatives Signal (−9.9s) |
| `precipitation` | ⭐ gering–mittel | F-WEAT-02 — r=0.036, Dosis-Wirkung messbar |
| `temperature` | ⭐ gering (nicht-linear) | F-WEAT-04 — r=0.018 |
| ~~`is_windy`~~ | ❌ entfernt | F-WEAT-03 — NaN, nie befüllt; aus Feature-Set entfernt |
| **Quantitatives Ranking** | 🔜 nach Modell-Training | Feature Importance / SHAP |

---

### Kernfrage 6 — Sind Verspätungen vorhersagbar?

**🔜 Modellierungs-Phase.** Aber: Analyse liefert starke Prior-Evidenz.

| Aspekt | Einschätzung | Basis |
|:---|:---|:---|
| Grundsätzliche Vorhersagbarkeit | ✅ sehr wahrscheinlich — starke zeitliche + räumliche Muster | F-TEMP-01/02, F-SPAT-01 |
| Bekannte Schwierigkeiten | Extremwerte (F-TARGET-07), Klassenungleichgewicht Events (F-EVNT-05), Linie E Outlier (F-TARGET-12) | — |
| Feature-Readiness | ⚠️ Feature Engineering in `02_preparation` noch ausstehend; `is_windy` zu debuggen | Phase C |
| Baseline-Metrik | OTP 87.0% → Random-Guess würde ~87% erreichen → Modell muss besser sein | F-TARGET |
| Modell-Kandidaten | GradientBoosting / LightGBM (Interaktionen) — 🔜 Modell-Phase | — |

## Setup

In [ ]:
from zh_tram_flow.notebook import *

TRAIN, TEST, lf = setup_analysis("03_analysis_0-overview")

%load_ext autoreload
%autoreload 2